In [1]:
cd /root/hardware_rounding_error_predictor
# --- pdf parser + symlink literature into /workspace so extract_pdf_text finds it
apt-get install -y poppler-utils
mkdir -p /workspace
ln -sfn /root/hardware_rounding_error_predictor/literature /workspace/literature

# --- caches on /root (not the flaky /workspace mount from previous pods) --------
export HF_HOME=/root/hf_cache
export MUFU_CACHE_DIR=/root/mufu_cache
mkdir -p /root/hf_cache /root/mufu_cache

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (24.02.0-1ubuntu9.8).
0 upgraded, 0 newly installed, 0 to remove and 60 not upgraded.


In [2]:
# --- build catalog for the 4 demo seq lengths -----------------------------------
python3 build_catalog.py --seq-lens 100 250 1000 4000 --no-verify 2>&1 | tee /root/catalog_build.log

M grid restricted to 4 specified seq lengths: [100, 250, 1000, 4000]

=== down_proj (N=2560, K=9728) ===
  [ 1/ 4] M=  100 N= 2560 K= 9728 /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
INFO:2026-04-23 14:40:45 3204:3204 init.cpp:119] If you see CUPTI_ERROR_INSUFFICIENT_PRIVILEGES, refer to https://developer.nvidia.com/nvidia-development-tools-solutions-err-nvgpuctrperm-cupti
  recipe=split_k_workspace_outtype    split_k=3  verify=skipped               kernel=none
  [ 2/ 4] M=  250 N= 2560 K= 9728   recipe=single_walk                  split_k=1  verify=skipped               kernel=none
  [ 3/ 4] M= 1000 N= 2560 K= 9728   recipe=single_walk                  split_k=1  verify=skipped               kernel=none
  [ 4/ 4] M= 4000 N= 2560 K= 9728   recipe=single_walk                 

In [3]:
# --- demo script --------------------------------------------------------------
cat > /root/demo.sh << 'EOF'
#!/bin/bash
cd /root/hardware_rounding_error_predictor
export HF_HOME=/root/hf_cache MUFU_CACHE_DIR=/root/mufu_cache
for SEQ in 100 250 1000 4000; do
    echo "================ seq=$SEQ ================"
    python3 ffn_chain_test.py all $SEQ
    echo "---- cuBLAS mode ----"
    EMULATOR_TARGET=cublas python3 ffn_chain_test.py compare $SEQ
done
EOF

# --- auto-detect SKU for the log filename ---------------------------------------
SKU=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | tr -d ' ')
echo "Logging demo to /root/ffn_demo_${SKU}.log"

bash /root/demo.sh 2>&1 | tee /root/ffn_demo_${SKU}.log

Logging demo to /root/ffn_demo_NVIDIAH100NVL.log
================ seq=100 ================
Sequence length override: 100
Data dir: ffn_chain_data/seq100
PHASE 1: INSTRUMENTED MODEL FORWARD PASS

Loading Qwen/Qwen3-4B...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 319.08it/s]
Loaded. 36 layers, targeting layer 20.

ARCHITECTURE INSPECTION
------------------------------------------------------------
  model.config.hidden_size:       2560
  model.config.intermediate_size:  9728
  model.config.hidden_act:         silu
  model.config.rms_norm_eps:       1e-06
  LayerNorm type:  Qwen3RMSNorm
  MLP type:        Qwen3MLP
  act_fn type:     SiLUActivation
  gate_proj:       torch.Size([9728, 2560])  bias=False
  up_proj:         torch.Size([9728, 2560])  bias=False
  down_proj:       torch.Size([2560, 9728])  bias=False
  ln weight shape: torch.Size([2560])

TOKENIZATION
------------------------------------------------